In [1]:
esperado = {"maçã", "banana", "uva"}
recebido = {"maçã", "uva", "kiwi"}

print(esperado - recebido)   # o que eu esperava e NÃO veio
print(recebido - esperado)   # o que veio e eu NÃO esperava
print(bool(set())) 

{'banana'}
{'kiwi'}
False


In [4]:
import pandas as pd
df = pd.read_csv("data/raw/dirty_cafe_sales.csv", dtype=str, keep_default_na=False)

print(df)

     Transaction ID      Item Quantity Price Per Unit Total Spent  \
0       TXN_1961373    Coffee        2            2.0         4.0   
1       TXN_4977031      Cake        4            3.0        12.0   
2       TXN_4271903    Cookie        4            1.0       ERROR   
3       TXN_7034554     Salad        2            5.0        10.0   
4       TXN_3160411    Coffee        2            2.0         4.0   
...             ...       ...      ...            ...         ...   
9995    TXN_7672686    Coffee        2            2.0         4.0   
9996    TXN_9659401                  3                        3.0   
9997    TXN_5255387    Coffee        4            2.0         8.0   
9998    TXN_7695629    Cookie        3                        3.0   
9999    TXN_6170729  Sandwich        3            4.0        12.0   

      Payment Method  Location Transaction Date  
0        Credit Card  Takeaway       2023-09-08  
1               Cash  In-store       2023-05-16  
2        Credit Card 

In [5]:
df["Item"].value_counts()

Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
             333
ERROR        292
Name: count, dtype: int64

In [6]:
df["Payment Method"].value_counts()

Payment Method
                  2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64

In [8]:
df["Location"].value_counts()

Location
            3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64

In [9]:
(df == "ERROR").sum()

Transaction ID        0
Item                292
Quantity            170
Price Per Unit      190
Total Spent         164
Payment Method      306
Location            358
Transaction Date    142
dtype: int64

In [11]:
pd.to_numeric(df["Quantity"], errors="coerce")

0       2.0
1       4.0
2       4.0
3       2.0
4       2.0
       ... 
9995    2.0
9996    3.0
9997    4.0
9998    3.0
9999    3.0
Name: Quantity, Length: 10000, dtype: float64

In [12]:
pd.to_numeric(df["Price Per Unit"], errors="coerce")

0       2.0
1       3.0
2       1.0
3       5.0
4       2.0
       ... 
9995    2.0
9996    NaN
9997    2.0
9998    NaN
9999    4.0
Name: Price Per Unit, Length: 10000, dtype: float64

In [13]:
pd.to_numeric(df["Total Spent"], errors="coerce")

0        4.0
1       12.0
2        NaN
3       10.0
4        4.0
        ... 
9995     4.0
9996     3.0
9997     8.0
9998     3.0
9999    12.0
Name: Total Spent, Length: 10000, dtype: float64

In [14]:
df.groupby("Item")["Price Per Unit"].unique()

Item
            [3.0, 2.0, 1.0, 5.0, , 4.0, 1.5, ERROR, UNKNOWN]
Cake                                 [3.0, , UNKNOWN, ERROR]
Coffee                               [2.0, , ERROR, UNKNOWN]
Cookie                               [1.0, UNKNOWN, , ERROR]
ERROR       [1.5, 3.0, 5.0, , 4.0, 2.0, 1.0, UNKNOWN, ERROR]
Juice                                [3.0, , UNKNOWN, ERROR]
Salad                                [5.0, ERROR, UNKNOWN, ]
Sandwich                             [4.0, , ERROR, UNKNOWN]
Smoothie                             [4.0, , UNKNOWN, ERROR]
Tea                                  [1.5, , ERROR, UNKNOWN]
UNKNOWN     [3.0, 1.0, 5.0, 4.0, 1.5, 2.0, , UNKNOWN, ERROR]
Name: Price Per Unit, dtype: object

In [15]:
df["Transaction ID"].is_unique

True

In [16]:
   df.isin({"", "ERROR", "UNKNOWN"}).sum()

Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [17]:
for coluna in ["Quantity", "Price Per Unit", "Total Spent"]:
    nan_gerados = pd.to_numeric(df[coluna], errors="coerce").isna().sum()
    sentinelas = df[coluna].isin({"", "ERROR", "UNKNOWN"}).sum()
    print(coluna, nan_gerados, sentinelas)

Quantity 479 479
Price Per Unit 533 533
Total Spent 502 502


In [18]:
qtd = pd.to_numeric(df["Quantity"], errors="coerce")
print(qtd.min(), qtd.max())

1.0 5.0


In [19]:
datas = pd.to_datetime(df["Transaction Date"], errors="coerce", format="%Y-%m-%d")
print(datas.isna().sum(), datas.min(), datas.max())

460 2023-01-01 00:00:00 2023-12-31 00:00:00


In [20]:
q = pd.to_numeric(df["Quantity"], errors="coerce")
p = pd.to_numeric(df["Price Per Unit"], errors="coerce")
t = pd.to_numeric(df["Total Spent"], errors="coerce")

completas = q.notna() & p.notna() & t.notna()
print("linhas completas:", completas.sum())
print("linhas onde a conta NÃO fecha:", (q[completas] * p[completas] != t[completas]).sum())

linhas completas: 8544
linhas onde a conta NÃO fecha: 0
